# ConnectX 自对弈 NNUE 训练 (v3: 温度采样 + 去重 + 验证早停 + 对抗赛)

**在 v2 加速版基础上的改进：**

1. **全程温度采样，取消随机开局**：每一手都对根节点 7 列的**精确搜索分值**做 softmax 温度采样（`SAMPLE_TEMP` 可调，默认 0.35）。局面多样性靠受控随机贯穿全局，而不是靠前几步乱下——避免了"确定性 argmax 导致对局轨迹收敛、数据大量重叠"的问题，同时从第一手起就产出训练标签。
   - **证明值与评估值严格分离**：搜索内部用 `±(100+depth)` 编码已证明的终局胜/负（`WIN_SCORE=100`），与 tanh 在 float32 下可能饱和到 ±1.0 的评估值不再混淆；深度奖励让 argmax 自动偏好最短胜路 / 最长抵抗；
   - 已证明必胜的局面直接走最优着法，保证终局兑现；
   - 已证明必败的着法被屏蔽不参与采样，避免无谓送子污染对局；
   - **训练标签始终取根局面的最优列分值**，写入前把证明值映射回 ±1.0（未证明的评估值本身在 (-1,1) 内原样保留），标签全部落在 tanh 值域 [-1,1]，与实际采样走了哪一步无关，标签质量不受探索影响。
2. **训练前按局面去重**：对称增广后按 `(player_mark, 42 格)` 分组、`value` 取均值。重复局面不会在 MSE loss 里被反复过采样、主导梯度。历史回放混合时同一局面优先保留新标签。
3. **探索深度默认 10**（`SEARCH_DEPTH` 可调）。
4. **历史数据回放（抗灾难性遗忘）**：每轮从历史池抽取 `HISTORY_RATIO` 与当轮新数据混合训练，超出 `MAX_HISTORY_ROWS` 时裁剪最旧数据。
5. **验证集 + 早停**：每轮训练留 10% 验证集，验证 Loss 连续 3 轮不降即停并**回滚到验证最优权重**（lr 降为 3e-4，最多 30 epoch）。自举标签有噪声下限，硬压训练 Loss 只会过拟合。
6. **对抗赛 + 灾难回滚门控**：每轮训练后，新模型 vs 训练前模型打 `2×ARENA_PAIRS` 局（默认 150 局：同一随机开局正反手各一局，新模型先后手各 75 局），抵消先手优势和开局不公；双方确定性 argmax，结果可复现。得分率 <50% 仅警告不回滚（容忍对抗赛统计噪声，AlphaZero 式无门控）；**得分率 <35% 或连续 2 轮 <45% 判定为灾难**：回滚到训练前模型，并把本轮数据移出历史回放池，防止坏数据复利污染。
7. **每轮编号 checkpoint**：每轮结束保存 `*_iterN.pth`（回滚轮保存恢复后的模型），训练崩溃可追溯到任意一轮恢复。

**继承 v2 的加速内核（数学等价原版搜索）：** 49 位位棋盘、Numba JIT、即胜预检、手写 NNUE 内积、多进程自对弈。

> 注意：为了给温度采样提供所有列的**精确**分值，根节点的每个子树用全窗口搜索（根层不做兄弟间 alpha 剪枝，子树内部剪枝不变），比单一最优列搜索略慢，属于必要开销。
> 网络结构与数据格式与原版一致。

In [ ]:
import os
import time
import math
import multiprocessing as mp
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from kaggle_environments import make

# Numba 是搜索加速的核心（Kaggle 镜像已预装；若无则自动安装）
try:
    from numba import njit
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numba'], check=True)
    from numba import njit

class ConnectX_NNUE(nn.Module):
    def __init__(self):
        super(ConnectX_NNUE, self).__init__()
        self.fc1 = nn.Linear(84, 256)
        self.fc2 = nn.Linear(256, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        return torch.tanh(self.fc4(x))

class ConnectXDataset(Dataset):
    def __init__(self, csv_file):
        self.data = pd.read_csv(csv_file)
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        player_mark = int(row['player_mark'])
        opp_mark = 2 if player_mark == 1 else 1
        board = row.iloc[1:43].values.astype(int)

        features = np.zeros(84, dtype=np.float32)
        features[:42] = (board == player_mark).astype(np.float32)
        features[42:] = (board == opp_mark).astype(np.float32)
        target = np.float32(row['value'])

        return torch.tensor(features), torch.tensor([target])

class NNUE_Evaluator:
    def __init__(self, model_path='nnue_model.pth'):
        self.model = ConnectX_NNUE()
        self.model.load_state_dict(torch.load(model_path, map_location='cpu'))
        self.model.eval()

        with torch.no_grad():
            # 注意: 这里统一加 .detach()，避免某些 torch 版本下 .numpy() 报错
            # 并保证 float32 + C 连续内存，供 Numba 高速内核直接使用
            self.w1 = np.ascontiguousarray(self.model.fc1.weight.detach().numpy().T, dtype=np.float32)
            self.b1 = np.ascontiguousarray(self.model.fc1.bias.detach().numpy(), dtype=np.float32)
            self.w2 = np.ascontiguousarray(self.model.fc2.weight.detach().numpy().T, dtype=np.float32)
            self.b2 = np.ascontiguousarray(self.model.fc2.bias.detach().numpy(), dtype=np.float32)
            self.w3 = np.ascontiguousarray(self.model.fc3.weight.detach().numpy().T, dtype=np.float32)
            self.b3 = np.ascontiguousarray(self.model.fc3.bias.detach().numpy(), dtype=np.float32)
            self.w4 = np.ascontiguousarray(self.model.fc4.weight.detach().numpy().T, dtype=np.float32)
            self.b4 = np.ascontiguousarray(self.model.fc4.bias.detach().numpy(), dtype=np.float32)

        # Numba 评估内核的共享临时缓冲（搜索为单线程，可安全复用）
        self.buf2 = np.zeros(64, dtype=np.float32)
        self.buf3 = np.zeros(32, dtype=np.float32)

    # 下面两个 numpy 方法保留用于兼容与自检；高速路径走 Numba 内核
    def get_initial_accumulator(self, board, player_mark):
        opp_mark = 2 if player_mark == 1 else 1
        features = np.zeros(84, dtype=np.float32)
        features[:42] = (board == player_mark).astype(np.float32)
        features[42:] = (board == opp_mark).astype(np.float32)
        return np.dot(features, self.w1) + self.b1

    def evaluate_from_accumulator(self, accumulator):
        x = np.maximum(0, accumulator)
        x = np.dot(x, self.w2) + self.b2
        x = np.maximum(0, x)
        x = np.dot(x, self.w3) + self.b3
        x = np.maximum(0, x)
        x = np.dot(x, self.w4) + self.b4
        return np.tanh(x)[0]

In [ ]:
import copy
from torch.utils.data import random_split

def train_model(csv_file, input_model_path=None, output_model_path='/kaggle/working/nnue_model.pth',
                epochs=30, batch_size=256, val_ratio=0.1, patience=3, lr=3e-4):
    if not os.path.exists(csv_file):
        print(f"数据文件 {csv_file} 不存在！")
        return

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"[{'GPU' if torch.cuda.is_available() else 'CPU'}] 开始训练模型...")

    model = ConnectX_NNUE().to(device)

    # 如果提供了输入模型路径且存在，则加载它作为预训练权重
    if input_model_path and os.path.exists(input_model_path):
        print(f"发现预训练模型 {input_model_path}，加载并继续训练...")
        model.load_state_dict(torch.load(input_model_path, map_location=device))
    else:
        print("未提供预训练模型或路径不存在，将从头开始训练...")

    dataset = ConnectXDataset(csv_file)

    # 划分验证集：监控过拟合并驱动早停；数据太少时退化为纯训练
    n_val = int(len(dataset) * val_ratio) if len(dataset) >= 20 and val_ratio > 0 else 0
    if n_val > 0:
        gen = torch.Generator().manual_seed(42)
        train_set, val_set = random_split(dataset, [len(dataset) - n_val, n_val], generator=gen)
        val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
        print(f"训练集 {len(train_set)} 条 / 验证集 {n_val} 条，早停耐心 {patience} 轮，lr={lr}")
    else:
        train_set, val_loader = dataset, None
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_val = float('inf')
    best_state = None
    bad_epochs = 0
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for features, targets in train_loader:
            features, targets = features.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
        train_loss = total_loss / len(train_loader)

        if val_loader is not None:
            model.eval()
            vloss = 0
            with torch.no_grad():
                for features, targets in val_loader:
                    features, targets = features.to(device), targets.to(device)
                    vloss += criterion(model(features), targets).item()
            vloss /= len(val_loader)
            print(f"Epoch {epoch+1}/{epochs} - 训练 Loss: {train_loss:.5f} / 验证 Loss: {vloss:.5f}")

            if vloss < best_val - 1e-5:
                best_val = vloss
                best_state = copy.deepcopy(model.state_dict())
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= patience:
                    print(f"验证 Loss 连续 {patience} 轮未改善，早停于第 {epoch+1} 轮（最佳验证 Loss {best_val:.5f}）")
                    break
        else:
            print(f"Epoch {epoch+1}/{epochs} - 平均 Loss: {train_loss:.5f}")

    if best_state is not None:
        model.load_state_dict(best_state) # 回滚到验证集最优权重，而非最后一轮
    torch.save(model.state_dict(), output_model_path)
    print(f"模型已保存到: {output_model_path}")

In [ ]:
# ======================================================================
# 高速搜索：49 位位棋盘 + Numba JIT
# 位棋盘编码: bit = col*7 + row (row 0 = 底部)，每列 7 bit（6 格 + 1 哨兵位）
# 特征索引与原版一致: feat = (5-row)*7 + col（observation.board 第 0 行是顶部）
#
# 证明值编码：|score| >= WIN_SCORE 表示已证明的终局胜/负（100 + 剩余depth，
# 剩余深度越大 = 离根越近 = 越快兑现，argmax 自动偏好速胜/最长抵抗）。
# NNUE 评估经 tanh 在 float32 下会饱和到恰好 ±1.0，因此必须与证明值严格分离。
# 写入训练标签前，证明值会被映射回 ±1.0，保证标签始终在 tanh 值域 [-1, 1] 内。
# ======================================================================
MOVE_ORDER = np.array([3, 2, 4, 1, 5, 0, 6], dtype=np.int64) # 中心优先的着法顺序
WIN_SCORE = 100.0   # 证明值门限：|score| >= WIN_SCORE 为已证明的胜/负
ILLEGAL = -1e9      # 非法列哨兵值（必须低于任何证明负值 -(100+depth)）
FEAT_OF_BIT = np.zeros(49, dtype=np.int64)
for _c in range(7):
    for _r in range(6):
        FEAT_OF_BIT[_c * 7 + _r] = (5 - _r) * 7 + _c

@njit
def _has_won(bb):
    # 竖 / 横 / 两种斜线
    m = bb & (bb >> 1)
    if m & (m >> 2): return True
    m = bb & (bb >> 7)
    if m & (m >> 14): return True
    m = bb & (bb >> 6)
    if m & (m >> 12): return True
    m = bb & (bb >> 8)
    if m & (m >> 16): return True
    return False

@njit
def _build_state(board, my_piece):
    # board: int64 数组 (42,)，第 0 行是顶部（同 observation.board）
    bb_me = 0
    bb_opp = 0
    npieces = 0
    heights = np.empty(7, np.int64) # 每列下一个空位的 bit 索引
    cnt = np.zeros(7, np.int64)
    for f in range(42):
        v = board[f]
        if v != 0:
            rt = f // 7
            c = f % 7
            bpos = c * 7 + (5 - rt)
            cnt[c] += 1
            npieces += 1
            if v == my_piece:
                bb_me |= (1 << bpos)
            else:
                bb_opp |= (1 << bpos)
    for c in range(7):
        heights[c] = 7 * c + cnt[c]
    return bb_me, bb_opp, heights, npieces

@njit
def _build_acc(bb_me, bb_opp, w1, b1, feat_of_bit):
    acc = b1.copy()
    for bpos in range(49):
        if (bb_me >> bpos) & 1:
            f = feat_of_bit[bpos]
            for k in range(acc.shape[0]):
                acc[k] += w1[f, k]
        elif (bb_opp >> bpos) & 1:
            f = feat_of_bit[bpos] + 42
            for k in range(acc.shape[0]):
                acc[k] += w1[f, k]
    return acc

@njit(fastmath=True)
def _nnue_eval(acc, w2, b2, w3, b3, w4, b4, buf2, buf3):
    # 等价于 relu(acc)@w2+b2 -> relu -> @w3+b3 -> relu -> @w4+b4 -> tanh
    for j in range(buf2.shape[0]): buf2[j] = b2[j]
    for i in range(acc.shape[0]):
        ai = acc[i]
        if ai > 0.0:
            for j in range(buf2.shape[0]):
                buf2[j] += ai * w2[i, j]
    for j in range(buf3.shape[0]): buf3[j] = b3[j]
    for i in range(buf2.shape[0]):
        hi = buf2[i]
        if hi > 0.0:
            for j in range(buf3.shape[0]):
                buf3[j] += hi * w3[i, j]
    s = b4[0]
    for i in range(buf3.shape[0]):
        hi = buf3[i]
        if hi > 0.0:
            s += hi * w4[i, 0]
    return math.tanh(s)

@njit(fastmath=True)
def _alphabeta(bb_me, bb_opp, heights, npieces, depth, alpha, beta, is_max,
               acc, w1, w2, b2, w3, b3, w4, b4, order, fob, buf2, buf3):
    if is_max:
        # 即胜预检：行棋方有立即获胜着法 -> 返回证明值（100+depth，越浅越大 = 偏好速胜）
        for oi in range(7):
            c = order[oi]; h = heights[c]
            if h < 7 * c + 6 and _has_won(bb_me | (1 << h)): return 100.0 + depth
        best = -1e30
        for oi in range(7):
            c = order[oi]; h = heights[c]
            if h >= 7 * c + 6: continue
            bb2 = bb_me | (1 << h)
            heights[c] = h + 1
            if npieces + 1 == 42:
                score = 0.0
            else:
                nacc = acc + w1[fob[h]] # 复制传递，保持与 numpy 参考实现一致的浮点路径
                if depth == 1:
                    score = _nnue_eval(nacc, w2, b2, w3, b3, w4, b4, buf2, buf3)
                else:
                    score = _alphabeta(bb2, bb_opp, heights, npieces + 1, depth - 1, alpha, beta, False,
                                       nacc, w1, w2, b2, w3, b3, w4, b4, order, fob, buf2, buf3)
            heights[c] = h
            if score > best: best = score
            if best > alpha: alpha = best
            if alpha >= beta: break
        return best
    else:
        for oi in range(7):
            c = order[oi]; h = heights[c]
            if h < 7 * c + 6 and _has_won(bb_opp | (1 << h)): return -(100.0 + depth)
        best = 1e30
        for oi in range(7):
            c = order[oi]; h = heights[c]
            if h >= 7 * c + 6: continue
            bb2 = bb_opp | (1 << h)
            heights[c] = h + 1
            if npieces + 1 == 42:
                score = 0.0
            else:
                nacc = acc + w1[fob[h] + 42]
                if depth == 1:
                    score = _nnue_eval(nacc, w2, b2, w3, b3, w4, b4, buf2, buf3)
                else:
                    score = _alphabeta(bb_me, bb2, heights, npieces + 1, depth - 1, alpha, beta, True,
                                       nacc, w1, w2, b2, w3, b3, w4, b4, order, fob, buf2, buf3)
            heights[c] = h
            if score < best: best = score
            if best < beta: beta = best
            if alpha >= beta: break
        return best

@njit(fastmath=True)
def search_root_scores(board, my_piece, depth,
                       w1, b1, w2, b2, w3, b3, w4, b4, order, fob, buf2, buf3):
    # 返回 7 列的精确根分值数组（非法列 = ILLEGAL=-1e9），供温度采样使用。
    # 每个根子结点用全窗口 (-inf, inf) 搜索：所有列的分值都是精确 minimax 值，
    # 不会被兄弟结点的 alpha 剪枝截断成边界值；子树内部仍正常 alpha-beta 剪枝。
    # 证明值 |score| >= 100 与 NNUE 评估值 (-1,1) 严格分离。
    bb_me, bb_opp, heights, npieces = _build_state(board, my_piece)
    acc = _build_acc(bb_me, bb_opp, w1, b1, fob)
    scores = np.full(7, -1e9)
    for oi in range(7):
        c = order[oi]; h = heights[c]
        if h >= 7 * c + 6: continue
        bb2 = bb_me | (1 << h)
        heights[c] = h + 1
        if _has_won(bb2):
            score = 100.0 + depth
        elif npieces + 1 == 42:
            score = 0.0
        else:
            nacc = acc + w1[fob[h]]
            if depth == 1:
                score = _nnue_eval(nacc, w2, b2, w3, b3, w4, b4, buf2, buf3)
            else:
                score = _alphabeta(bb2, bb_opp, heights, npieces + 1, depth - 1, -1e30, 1e30, False,
                                   nacc, w1, w2, b2, w3, b3, w4, b4, order, fob, buf2, buf3)
        heights[c] = h
        scores[c] = score
    return scores

# ---------------- 自对弈 Agent：全程温度采样，无随机开局 ----------------
evaluator = None
training_data = []
SEARCH_DEPTH = 10   # 探索深度
SAMPLE_TEMP = 0.35  # 采样温度：越大探索越强，越小越接近 argmax

def get_augmented_data(player_mark, board, value):
    row_orig = [player_mark] + list(board) + [value]
    grid = np.array(board).reshape(6, 7)
    row_flipped = [player_mark] + np.fliplr(grid).flatten().tolist() + [value]
    return [row_orig, row_flipped]

def choose_action_by_temperature(scores):
    # 返回 (采样着法, 训练标签)。
    # 标签取最优列的分值并映射回 tanh 值域：证明胜 -> 1.0，证明负 -> -1.0，
    # 未证明的评估值 (-1,1) 原样保留。与实际采样走哪一步无关。
    valid = [int(c) for c in MOVE_ORDER if scores[c] > -1e8]
    best_col = valid[0]
    for c in valid:
        if scores[c] > scores[best_col]: best_col = c
    best = float(scores[best_col])

    if best >= WIN_SCORE:
        return best_col, 1.0    # 已证明必胜：直接兑现（深度奖励保证走最短胜路），标签 1.0

    # 屏蔽已证明必败的着法，避免无谓送子污染对局
    cand = [c for c in valid if scores[c] > -WIN_SCORE]
    if not cand:
        # 所有着法都已证明必败：走"最长抵抗"（证明值最大 = 输得最慢），标签 -1.0
        return best_col, -1.0

    # 此时 best 一定是未证明的评估值，落在 (-1, 1)，可直接作为标签
    logits = np.array([(float(scores[c]) - best) / SAMPLE_TEMP for c in cand])
    probs = np.exp(logits)
    probs /= probs.sum()
    action = int(np.random.choice(cand, p=probs))
    return action, best

def self_play_agent(observation, configuration):
    try:
        board, my_piece = np.array(observation.board), observation.mark

        scores = search_root_scores(
            board.astype(np.int64), my_piece, SEARCH_DEPTH,
            evaluator.w1, evaluator.b1, evaluator.w2, evaluator.b2,
            evaluator.w3, evaluator.b3, evaluator.w4, evaluator.b4,
            MOVE_ORDER, FEAT_OF_BIT, evaluator.buf2, evaluator.buf3)

        action, value = choose_action_by_temperature(scores)
        # 从第一手起就记录标签（原来随机开局阶段没有任何训练数据）
        training_data.extend(get_augmented_data(my_piece, board, value))

        return int(action)
    except Exception as e:
        print(f"Self-play Error: {e}")
        return int(random.choice([c for c in range(7) if observation.board[c] == 0]))

# 每个子进程独立持有 evaluator、临时缓冲和 training_data，避免共享可变状态。
def _init_selfplay_worker(model_path):
    global evaluator, training_data
    torch.set_num_threads(1)  # 避免多个进程各自启动大量 PyTorch 线程
    evaluator = NNUE_Evaluator(model_path)
    training_data = []

def _play_games_worker(task):
    worker_id, games, seed = task
    global training_data
    random.seed(seed)
    np.random.seed(seed % (2**32 - 1))
    training_data = []
    env = make("connectx", debug=False)
    for _ in range(games):
        env.reset()
        env.run([self_play_agent, self_play_agent])
    return worker_id, games, training_data


In [ ]:
import shutil

history_pool = [] # 历史数据池：元素为历轮 DataFrame，跨迭代累积
arena_low_streak = 0 # 对抗赛连续低分（<45%）轮数，用于灾难回滚判定
BOARD_COLS = ['player_mark'] + [f'cell_{i}' for i in range(42)] # 用于按局面去重的键

# ---------------- 对抗赛评估：随机开局镜像，正反手各一局 ----------------
def _random_opening(rng, plies):
    cnt = [0] * 7
    seq = []
    for _ in range(plies):
        c = int(rng.choice([col for col in range(7) if cnt[col] < 6]))
        seq.append(c)
        cnt[c] += 1
    return seq

def _greedy_col(scores):
    # 非法列哨兵为 -1e9，证明负值约 -(100+depth)，判断阈值须区分两者
    valid = [int(c) for c in MOVE_ORDER if scores[c] > -1e8]
    best = valid[0]
    for c in valid:
        if scores[c] > scores[best]: best = c
    return best

def _arena_game(ev_p1, ev_p2, opening, depth):
    # 返回 1=先手胜, 2=后手胜, 0=平。双方均确定性 argmax，同开局下结果可复现
    board = np.zeros(42, dtype=np.int64)
    mark, plies = 1, 0
    for c in opening: # 开局最多 6 步，双方各 ≤3 子，不可能提前分出胜负
        for r in range(5, -1, -1):
            if board[r * 7 + c] == 0:
                board[r * 7 + c] = mark
                break
        plies += 1
        mark = 3 - mark
    while plies < 42:
        ev = ev_p1 if mark == 1 else ev_p2
        scores = search_root_scores(board, mark, depth,
                                    ev.w1, ev.b1, ev.w2, ev.b2, ev.w3, ev.b3, ev.w4, ev.b4,
                                    MOVE_ORDER, FEAT_OF_BIT, ev.buf2, ev.buf3)
        c = _greedy_col(scores)
        for r in range(5, -1, -1):
            if board[r * 7 + c] == 0:
                board[r * 7 + c] = mark
                break
        plies += 1
        bb_me, _, _, _ = _build_state(board, mark)
        if _has_won(bb_me):
            return mark
        mark = 3 - mark
    return 0

def arena_evaluate(new_model_path, old_model_path, num_pairs=20, depth=8, opening_plies=4, seed=0):
    # 每个随机开局打两局：新旧模型正反手交换，抵消先手优势与开局不公
    ev_new, ev_old = NNUE_Evaluator(new_model_path), NNUE_Evaluator(old_model_path)
    rng = np.random.RandomState(seed)
    w = d = l = 0
    for _ in range(num_pairs):
        opening = _random_opening(rng, opening_plies)
        for new_first in (True, False):
            r = _arena_game(ev_new if new_first else ev_old,
                            ev_old if new_first else ev_new, opening, depth)
            if r == 0: d += 1
            elif (r == 1) == new_first: w += 1
            else: l += 1
    score = (w + 0.5 * d) / (2 * num_pairs)
    return w, d, l, score

def run_iteration(iteration=1, num_games=50, current_model_path='/kaggle/working/nnue_model.pth',
                  history_ratio=0.08, max_history_rows=2_000_000, num_workers=None,
                  arena_pairs=20, arena_depth=8, arena_opening_plies=4,
                  disaster_score=0.35, low_score=0.45, low_streak_limit=2):
    global evaluator, training_data, history_pool, arena_low_streak
    print(f"\n========== 迭代强化学习 第 {iteration} 轮 ==========")

    # 1. 加载当前最新模型用于自对弈
    try:
        evaluator = NNUE_Evaluator(current_model_path)
    except Exception as e:
        print(f"加载模型失败: {e}")
        print("请确保已经有初始模型！")
        return

    # JIT 预热：首次调用触发编译（几秒），避免计进对弈耗时
    if iteration == 1:
        _t0 = time.time()
        search_root_scores(np.zeros(42, dtype=np.int64), 1, 2,
                           evaluator.w1, evaluator.b1, evaluator.w2, evaluator.b2,
                           evaluator.w3, evaluator.b3, evaluator.w4, evaluator.b4,
                           MOVE_ORDER, FEAT_OF_BIT, evaluator.buf2, evaluator.buf3)
        print(f"Numba JIT 预热完成 ({time.time() - _t0:.1f}s)")

    # 2. 自对弈生成数据（全程温度采样保证局面多样性）
    if num_games < 1:
        raise ValueError("num_games 必须大于 0")
    if num_workers is None:
        # Kaggle CPU 实例通常只有少量 vCPU；默认上限 4，避免抢占训练所需资源。
        num_workers = min(4, os.cpu_count() or 1, num_games)
    num_workers = max(1, min(int(num_workers), num_games))
    training_data = []

    print(f"开始使用 NNUE 进行 {num_workers} 个进程的自对弈收集数据，共 {num_games} 局...")
    _t0 = time.time()
    if num_workers == 1:
        # 单进程路径便于小规模调试，也避免不支持 fork 的环境出错。
        _, _, training_data = _play_games_worker((0, num_games, iteration * 1_000_000))
    else:
        # Kaggle 运行在 Linux；fork 让 notebook 中已定义的 agent/Numba 函数可直接被子进程使用。
        try:
            ctx = mp.get_context("fork")
        except ValueError:
            print("当前环境不支持 fork，自动回退为单进程。")
            _, _, training_data = _play_games_worker((0, num_games, iteration * 1_000_000))
        else:
            base_games, remainder = divmod(num_games, num_workers)
            tasks = [(worker_id, base_games + (worker_id < remainder),
                      iteration * 1_000_000 + worker_id)
                     for worker_id in range(num_workers)]
            completed = 0
            with ctx.Pool(processes=num_workers, initializer=_init_selfplay_worker,
                          initargs=(current_model_path,)) as pool:
                for _, games, rows in pool.imap_unordered(_play_games_worker, tasks):
                    training_data.extend(rows)
                    completed += games
                    print(f"完成自对弈 {completed}/{num_games} 局...")
    elapsed = time.time() - _t0
    print(f"自对弈耗时 {elapsed:.1f}s，平均每局 {elapsed/num_games*1000:.0f}ms")

    # 3. 按局面去重（同一局面取 value 均值），再保存到 working 目录
    cols = BOARD_COLS + ['value']
    df_raw = pd.DataFrame(training_data, columns=cols)
    df_new = df_raw.groupby(BOARD_COLS, as_index=False)['value'].mean()
    new_csv = f"/kaggle/working/data_v{iteration+1}.csv"
    df_new.to_csv(new_csv, index=False)
    dup_rate = 1 - len(df_new) / max(len(df_raw), 1)
    print(f"生成数据 {len(df_raw)} 条，去重后 {len(df_new)} 条（重复率 {dup_rate:.1%}），已保存: {new_csv}")

    # 4. 历史数据回放：从历史池抽取 history_ratio 与新数据混合（同一局面优先用新标签）
    df_train = df_new
    train_csv = new_csv
    if history_ratio > 0 and history_pool:
        hist_all = pd.concat(history_pool, ignore_index=True)
        n_hist = int(len(hist_all) * history_ratio)
        if n_hist > 0:
            df_hist = hist_all.sample(n=n_hist, random_state=iteration)
            df_train = (pd.concat([df_new, df_hist], ignore_index=True)
                        .drop_duplicates(subset=BOARD_COLS, keep='first') # 新数据在前，冲突时保留新标签
                        .sample(frac=1.0, random_state=iteration)
                        .reset_index(drop=True))
            train_csv = f"/kaggle/working/train_mix_v{iteration+1}.csv"
            df_train.to_csv(train_csv, index=False)
            print(f"历史回放: 新数据 {len(df_new)} 条 + 历史 {n_hist}/{len(hist_all)} 条 "
                  f"({history_ratio:.0%}) -> 去重后训练集共 {len(df_train)} 条 ({train_csv})")

    # 新数据（已去重）入池；超出容量时裁剪最旧数据
    history_pool.append(df_new)
    while len(history_pool) > 1 and sum(len(d) for d in history_pool) > max_history_rows:
        history_pool.pop(0)

    # 5. 备份训练前的模型，供赛后对比
    prev_model_path = current_model_path + '.prev'
    shutil.copy(current_model_path, prev_model_path)

    # 6. 微调训练（验证集 + 早停，回滚到验证最优权重）
    print("开始利用新数据对 NNUE 进行微调升级...")
    train_model(train_csv, input_model_path=current_model_path,
                output_model_path=current_model_path, epochs=30, batch_size=256)

    # 7. 对抗赛 + 灾难回滚门控：容忍噪声级低分，只拦截真正的崩溃
    if arena_pairs > 0:
        _t0 = time.time()
        w, d, l, score = arena_evaluate(current_model_path, prev_model_path,
                                        num_pairs=arena_pairs, depth=arena_depth,
                                        opening_plies=arena_opening_plies, seed=10_000 + iteration)
        print(f"对抗赛({arena_pairs*2}局, 深度{arena_depth}, 镜像开局{arena_opening_plies}步): "
              f"新模型 胜{w} 平{d} 负{l}，得分率 {score:.1%}（{time.time()-_t0:.0f}s）")

        arena_low_streak = arena_low_streak + 1 if score < low_score else 0
        if score < disaster_score or arena_low_streak >= low_streak_limit:
            print(f"!!! 灾难回滚: 得分率 {score:.1%}"
                  f"（连续 {arena_low_streak} 轮 <{low_score:.0%} / 灾难线 {disaster_score:.0%}），"
                  f"恢复训练前模型并丢弃本轮数据")
            shutil.copy(prev_model_path, current_model_path)
            if history_pool and history_pool[-1] is df_new:
                history_pool.pop() # 本轮数据由被否决的训练流程产出，移出回放池
            arena_low_streak = 0
        elif score < 0.5:
            print(f"警告: 新模型得分率 {score:.1%} 未过半，未达灾难阈值，继续训练（低分连击 {arena_low_streak}）")

    # 8. 保存本轮接受的模型为带编号 checkpoint（回滚轮保存的是恢复后的模型），崩溃可追溯
    root, ext = os.path.splitext(current_model_path)
    ckpt_path = f"{root}_iter{iteration}{ext}"
    shutil.copy(current_model_path, ckpt_path)
    print(f"checkpoint 已保存: {ckpt_path}")

    print(f"第 {iteration} 轮迭代完成！")

In [ ]:
import shutil

# 【配置区】请根据你在 Kaggle 上的实际路径修改这里
# 预训练模型路径 (只读，你已有的具备一定棋力的模型)
INPUT_MODEL = '/kaggle/input/connectx-nnue/nnue_model_pretrained.pth'

# 训练输出的模型路径 (可写，自对弈迭代的当前模型)
OUTPUT_MODEL = '/kaggle/working/nnue_model.pth'

# 历史回放配置
HISTORY_RATIO = 0.04 # 每轮训练时从历史池抽取的比例
MAX_HISTORY_ROWS = 2_000_000 # 历史池容量上限，超出裁剪最旧数据

# 并行自对弈进程数：建议不超过 Kaggle 的可用 CPU 核数；设为 1 可切回串行。
SELFPLAY_WORKERS = min(4, os.cpu_count() or 1)

# 对抗赛配置：每轮训练后新旧模型互搏，衡量是否真的变强
ARENA_PAIRS = 75         # 随机开局的数量；每个开局正反手各打 1 局，共 150 局（新模型先后手各 75 局）
ARENA_DEPTH = 8          # 对抗赛双方的搜索深度（确定性 argmax）
ARENA_OPENING_PLIES = 4  # 随机开局步数（≤6 步不可能提前分出胜负）

# 【可选】如果历史数据想包含最初的人工/预训练数据集，填它的 CSV 路径；不需要则保持 None
INITIAL_HISTORY_CSV = None # 例: '/kaggle/input/connectx-nnue/selfplay_data.csv'

# 1. 准备初始模型：直接复制预训练模型，跳过繁琐的初始训练
if not os.path.exists(OUTPUT_MODEL):
    if os.path.exists(INPUT_MODEL):
        print(f"发现预训练模型 {INPUT_MODEL}，直接复制为初始模型开始自对弈...")
        shutil.copy(INPUT_MODEL, OUTPUT_MODEL)
    else:
        print(f"找不到预训练模型 {INPUT_MODEL}，请检查路径！")

# 2.【可选】把初始数据集放入历史池（首轮即可参与回放）
if INITIAL_HISTORY_CSV and os.path.exists(INITIAL_HISTORY_CSV):
    history_pool.append(pd.read_csv(INITIAL_HISTORY_CSV))
    print(f"初始历史数据已入池: {INITIAL_HISTORY_CSV}，共 {len(history_pool[0])} 条")

# 3. 直接开始迭代自对弈训练
if os.path.exists(OUTPUT_MODEL):
    for i in range(1, 31): # 迭代多轮
        run_iteration(iteration=i, num_games=300, current_model_path=OUTPUT_MODEL,
                      history_ratio=HISTORY_RATIO, max_history_rows=MAX_HISTORY_ROWS,
                      num_workers=SELFPLAY_WORKERS,
                      arena_pairs=ARENA_PAIRS, arena_depth=ARENA_DEPTH,
                      arena_opening_plies=ARENA_OPENING_PLIES)